# Примитивный ETL pipeline, исследование API Steam

### Вводное

In [1]:
import requests
import json
import pandas as pd
import time
import ijson
from itertools import islice
from pathlib import Path
import glob



In [2]:
domain = f"https://api.steampowered.com"
keyApi = "67000A785A07083160056E8B303C4E85"

In [3]:
def getSteamApi(interface, method, version):
    objectRequest = requests.get(url=domain + f"/{interface}/{method}/v{version}")
    return objectRequest


def getSteamApiPars(interface, method, version, pars: dict):
    objectRequest = requests.get(url=domain + f"/{interface}/{method}/v{version}", params=pars)
    return objectRequest


## Интерфейсы и методы

### Запрос интерфейсов

In [ ]:
apiRequest = getSteamApiPars("ISteamWebAPIUtil", "GetSupportedAPIList", "1", {"key": keyApi})
print(apiRequest.status_code)

### Список GET интерфейсов и методов

In [ ]:
interfaces = apiRequest.json()["apilist"]["interfaces"]  # список интерфейсов и их методов

interfacesGet = dict()  # get интерфейсы и методы, ключ: название интерфейса, значение: список методов

for interface in interfaces:
    interfaceName = interface["name"]
    for method in interface["methods"]:
        if "httpmethod" in method and method["httpmethod"] == "GET":
            if interfaceName in interfacesGet:
                interfacesGet[interfaceName].append(method)
            else:
                interfacesGet[interfaceName] = [method]

interfacesGetList = interfacesGet.keys()


In [ ]:
print(interfacesGetList)
print(len(interfacesGetList))

In [ ]:
interfaceName = "IStoreService"

interfaceCurrent = interfacesGet[interfaceName]

print(pd.DataFrame(interfaceCurrent))


## Extract

### Все доступные игр в Steam

In [ ]:
# https://partner.steamgames.com/doc/webapi/IStoreService#GetAppList

json_params = {
    "max_results": 50000
}

params = {
    "key": keyApi,
    "input_json": json.dumps(json_params)
}

appList = list()

while True:
    requestAppList = getSteamApiPars("IStoreService", "GetAppList", "1", pars=params).json()["response"]

    appList.extend(requestAppList.get("apps", []))

    if not requestAppList.get("have_more_results", False):
        break

    if "last_appid" in requestAppList:
        json_params["last_appid"] = requestAppList["last_appid"]
        params["input_json"] = json.dumps(json_params)



Сохраняем данные в row

In [ ]:
with open("../data/row/steam_games.json", 'w', encoding="utf-8") as f:
    json.dump(appList, f, indent=4, ensure_ascii=False)

with open("../data/row/steam_games.json", 'r', encoding="utf-8") as f:
    print(json.load(f))


## Расширенная информация по каждой игре Steam

In [16]:
with open("../data/row/steam_games.json", "r", encoding="utf-8") as f:
    countAllGames = len([i for i in ijson.items(f, "item")])


In [4]:
domain_store = "https://store.steampowered.com"

folderDataRow = Path("../data/row")
folderErrors = folderDataRow / "errors"
folderGameInfo = folderDataRow / "steam_game_info"

pathError = Path(folderErrors / "steam_error_requests.jsonl")
pathSteamGames = Path(folderDataRow / "steam_games.json")
pathBatch = lambda start, end: Path(folderGameInfo / f"batch_{start}_{end}.jsonl")


In [ ]:
finishedCount = 0
errorCount = 0

with open(pathError, "r", encoding="utf-8") as fError:
    for error in fError:
        errorCount += 1

for pathBatchCounter in Path(folderDataRow / "steam_game_info").glob("batch_*_*.jsonl"):
    with open(pathBatchCounter, "r", encoding="utf-8") as fBatch:
        for gInfo in fBatch:
            finishedCount += 1

start = finishedCount + errorCount
batchSize = 5000
end = 5000 * (start // 5000 + 1)

print(f"Всего обработано к моменту начала скрейпинга: {start}, {round((start / countAllGames) * 100, 2)} %")

with open(pathSteamGames, 'rb') as fRead, open(pathError, "a") as fErrors:
    current_batch_start = batchSize * ((start - errorCount) // batchSize)
    current_batch_end = current_batch_start + batchSize

    with open(pathBatch(current_batch_start, current_batch_end), "a", encoding="utf-8") as fWrite:

        batch = islice(ijson.items(fRead, "item"), start, end)
        for game in batch:
            appid = game["appid"]

            try:
                request = requests.get(domain_store + f"/api/appdetails?appids={appid}&cc=us&l=english", timeout=5)

                if request.status_code == 429:
                    time.sleep(100)
                    fErrors.write(json.dumps({appid: f"API success: False"}, ensure_ascii=False) + "\n")
                    continue
                elif request.status_code != 200:
                    fErrors.write(json.dumps({appid: request.status_code}, ensure_ascii=False) + "\n")
                    continue

                reqGame = request.json()[str(appid)]

                if reqGame["success"]:
                    gameInfo = reqGame["data"]
                    fWrite.write(json.dumps(gameInfo, ensure_ascii=False) + "\n")
                else:
                    fErrors.write(json.dumps({appid: f"API success: False"}, ensure_ascii=False) + "\n")

            except Exception as e:
                fErrors.write(json.dumps({appid: str(e)}, ensure_ascii=False) + "\n")
                continue

            time.sleep(1)

print(f"Батч отработан успешно")


Всего обработано к моменту начала скрейпинга: 21128, 11.44 %


In [15]:
with open(Path(folderGameInfo / "batch_20000_25000.jsonl"), "rb") as f:
    print(json.loads(next(f)).keys())


dict_keys(['type', 'name', 'steam_appid', 'required_age', 'is_free', 'detailed_description', 'about_the_game', 'short_description', 'supported_languages', 'header_image', 'capsule_image', 'capsule_imagev5', 'website', 'pc_requirements', 'mac_requirements', 'linux_requirements', 'developers', 'publishers', 'price_overview', 'packages', 'package_groups', 'platforms', 'categories', 'genres', 'screenshots', 'movies', 'release_date', 'support_info', 'background', 'background_raw', 'content_descriptors', 'ratings'])
